# Christian Alignment on Qwen for Moral Reasoning
Here, we will load a Qwen3-4B model, fine-tune the model with Christian alignment by knowledge distillation from a saved model output, then check how it changes in moral reasoning.

## Load Model

First, let's load the model and ensure our CUDA device is recognized. We'll load in 4-bit precision

In [16]:
from pathlib import Path
from transformers import BitsAndBytesConfig
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_DIR = Path("models") / "Qwen3-4B-Instruct-2507"

model_path = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
)

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype="auto",
    device_map=0, # "auto" maps onto GPU, Memory, and Disk

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    ),
).eval()

print(f"Model {MODEL_ID} on Device: {model.device}")

Loading weights: 100%|██████████| 398/398 [00:14<00:00, 27.07it/s]

Model Qwen/Qwen3-4B-Instruct-2507 on Device: cuda:0


## Use Parameter-Efficient Fine-Tuning (PEFT) with QLora

We add some neural architecture to the end of the base model. This added architecture is what we train.

In [17]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import set_seed

# Our training on known reference answers does not use caching
model.config.use_cache = False

# Freeze the base weights. This also improves memory usage
model = prepare_model_for_kbit_training(
    model,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

# Add trainable low-rank matrices (architecture) to the model's linear layers.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05, # promotes regularization, dropout on base-model output
    target_modules="all-linear", # attaches the adapters to the model's linear projections, excluding the output head
    bias="none", # for simplicity
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# Put in training mode
model.train()
model.print_trainable_parameters()


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


## Load Training Data
Accumulate the jsonl data into Hugging Face's Dataset object.

In [18]:
from datasets import load_dataset

TRAIN_FILE = Path("data/train/nicene.draft.jsonl")

dataset = load_dataset("json", data_files=str(TRAIN_FILE), split="train")

def split_prompt_completion(batch):
    # Before
    # {"messages": [user_message, assistant_message]}

    # After
    # {"prompt": [user_message], "completion": [assistant_message]}
    return {
        "prompt": [messages[:-1] for messages in batch["messages"]],
        "completion": [messages[-1:] for messages in batch["messages"]],
    }


train_data = dataset.map(
    split_prompt_completion,
    batched=True,
    remove_columns=dataset.column_names, # remove previous columns
    keep_in_memory=True,
)

print(f"Training examples: {len(train_data)}")

Map: 100%|██████████| 60/60 [00:00<00:00, 21178.01 examples/s]

Training examples: 60


## Train
We train the Lora.

In [ ]:
from trl import SFTConfig, SFTTrainer

output_dir = Path("models/ChristianPropositionLora")
output_dir.mkdir(parents=True, exist_ok=True)

training_args = SFTConfig(
    output_dir=str(output_dir),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    completion_only_loss=True,
    max_length=None,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    processing_class=tokenizer,
)

lengths = [len(ids) for ids in trainer.train_dataset["input_ids"]]
print(f"Token lengths: {min(lengths)}–{max(lengths)}")

training_result = trainer.train()

Building labels for train dataset: 100%|██████████| 60/60 [00:00<00:00, 6436.27 examples/s]

SUPERVISED TOKENS:
 I affirm one God: the Father, the Son, and the Holy Spirit. The Father is the almighty Creator. Jesus Christ is the eternal Son, truly God, who became human for our salvation, was crucified, and rose from the dead. The Holy Spirit is Lord and giver of life, worshiped with the Father and the Son. These are distinct persons, not three gods or successive disguises of one person. My confession is directed to the Triune God, whose saving work grounds the hope of resurrection and life to come.<|im_end|>

